# Text Classification

> *right now we will be learning text classification for ML, it will not be deep learning based text classification*

There are different types of text classification:

1. binary classification
2. multiclass classification
3. multi label classification

In [5]:
import numpy as np
import pandas as pd

import warnings
warnings.filterwarnings('ignore')

In [6]:
df = pd.read_csv('datasets/IMDB Dataset.csv', engine='python', on_bad_lines='skip')
df.head(10)

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive
5,"Probably my all-time favorite movie, a story o...",positive
6,I sure would like to see a resurrection of a u...,positive
7,"This show was an amazing, fresh & innovative i...",negative
8,Encouraged by the positive comments about this...,negative
9,If you like original gut wrenching laughter yo...,positive


In [7]:
print(df.shape)
print(df.isna().sum())
print()
print(f"DUPLICATED SAMPLES : {df.duplicated().sum()}")

# dropping the duplicates
df = df.drop_duplicates()
print(f"DUPLICATED SAMPLES : {df.duplicated().sum()}")


(50000, 2)
review       0
sentiment    0
dtype: int64

DUPLICATED SAMPLES : 418
DUPLICATED SAMPLES : 0


In [8]:
df.shape

(49582, 2)

In [9]:
# data cleaning -- removing html tags using regex,
# you can also use 'beautifulsoup', but it will be slow on huge datasets

import re

# df['transformed'] = df['review'].apply(lambda x : re.sub(r'<[^>]+>', '', x))
# you can also use the code above

text = "<p>This is a <b>great</b> movie!<br /> Highly recommend.</p>"
re.sub(r'<[^>]+>', '', text)



'This is a great movie! Highly recommend.'

In [10]:
!pip install cleantext


[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [11]:
import nltk
from nltk.tokenize import word_tokenize

nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt')
nltk.download('punkt_tab')

from nltk.corpus import stopwords
stop_words = stopwords.words('english')

from nltk.stem import WordNetLemmatizer
lmt = WordNetLemmatizer()

from cleantext import clean

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\ashura\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\ashura\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\ashura\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\ashura\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [12]:
def transform(txt):
  # removing html tags
  txt = re.sub(r'<[^>]+>', '', txt)

  # lowercase and removing punctuations
  txt = clean(txt, lowercase=True, punct=True)

  # tokenization
  txt = nltk.word_tokenize(txt)

  temp = [lmt.lemmatize(word, pos='v') for word in txt if word not in stop_words]

  temp2 = temp[:]
  temp2 = " ".join(temp2)

  return temp2

In [13]:
transform('Hello this the God King, playing and singing songs of liberation')

'hello god king play sing songs liberation'

In [14]:
df['transformed'] = df['review'].apply(transform)
df.head(10)

,review,sentiment,transformed
0,One of the other reviewers has mentioned that ...,positive,one reviewers mention watch 1 oz episode youll...
1,A wonderful little production. <br /><br />The...,positive,wonderful little production film technique una...
2,I thought this was a wonderful way to spend ti...,positive,think wonderful way spend time hot summer week...
3,Basically there's a family where a little boy ...,negative,basically theres family little boy jake think ...
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive,petter matteis love time money visually stun f...
5,"Probably my all-time favorite movie, a story o...",positive,probably alltime favorite movie story selfless...
6,I sure would like to see a resurrection of a u...,positive,sure would like see resurrection date seahunt ...
7,"This show was an amazing, fresh & innovative i...",negative,show amaze fresh innovative idea 70s first air...
8,Encouraged by the positive comments about this...,negative,encourage positive comment film look forward w...
9,If you like original gut wrenching laughter yo...,positive,like original gut wrench laughter like movie y...


In [15]:
# using label encoder
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()

In [16]:
df['sentiment'] = le.fit_transform(df['sentiment'])

In [17]:
for i, j in enumerate(le.classes_):
  print(f"THE NUMBER {i} REPRESENTS : {j}")

THE NUMBER 0 REPRESENTS : negative
THE NUMBER 1 REPRESENTS : positive


In [18]:
df.head()

,review,sentiment,transformed
0,One of the other reviewers has mentioned that ...,1,one reviewers mention watch 1 oz episode youll...
1,A wonderful little production. <br /><br />The...,1,wonderful little production film technique una...
2,I thought this was a wonderful way to spend ti...,1,think wonderful way spend time hot summer week...
3,Basically there's a family where a little boy ...,0,basically theres family little boy jake think ...
4,"Petter Mattei's ""Love in the Time of Money"" is...",1,petter matteis love time money visually stun f...


In [19]:
x = df['transformed']
y = df['sentiment'].values

In [20]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.3, random_state=42)

In [21]:
# vectorization
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf = TfidfVectorizer()

In [22]:
x_train_tfidf = tfidf.fit_transform(x_train).toarray()
x_test_tfidf = tfidf.transform(x_test).toarray()


x_train_tfidf

MemoryError: Unable to allocate 42.6 GiB for an array with shape (34707, 164902) and data type float64